# Chapter 10 Lab: Frames and Erasures

This notebook accompanies **Chapter 10**. It builds the erasure-robustness
toolkit (`subframe_spans`, `erasure_number`, `reconstruct_after_erasure`,
`tight_erasure_error_formula`) plus the Chapter 9 ETF checker, then works
through all five lab exercises.

Run the setup cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline

def frame_operator(F):
    return F @ F.T

def exact_frame_bounds(F):
    S = frame_operator(F)
    eigenvalues, eigenvectors = np.linalg.eigh(S)
    return float(eigenvalues[0]), float(eigenvalues[-1]), eigenvalues, eigenvectors

def regular_polygon(m):
    angles = 2 * np.pi * np.arange(m) / m
    return np.vstack([np.cos(angles), np.sin(angles)])

def gram_matrix(F):
    return F.conj().T @ F

def coherence(F):
    G = gram_matrix(F)
    off_diag = np.abs(G - np.diag(np.diag(G)))
    return off_diag.max()

def welch_bound(n, m):
    return np.sqrt((m - n) / (n * (m - 1)))

def is_etf(F, tol=1e-6):
    n, m = F.shape
    S = F @ F.T.conj()
    eigs = np.linalg.eigvalsh(S.real)
    tight = bool(np.isclose(eigs.min(), eigs.max(), atol=tol))
    G = gram_matrix(F)
    off = np.array([np.abs(G[i,j]) for i in range(m) for j in range(m) if i != j])
    equi = bool(np.isclose(off.max(), off.min(), atol=tol))
    mu = coherence(F)
    achieves = bool(np.isclose(mu, welch_bound(n, m), atol=tol))
    return tight, equi, achieves

def subframe_spans(F, erased_indices):
    n, m = F.shape
    kept = [i for i in range(m) if i not in erased_indices]
    if len(kept) < n:
        return False
    return np.linalg.matrix_rank(F[:, kept]) == n

def erasure_number(F):
    n, m = F.shape
    for k in range(1, m - n + 2):
        all_span = all(subframe_spans(F, list(E)) for E in combinations(range(m), k))
        if not all_span:
            return k - 1
    return m - n

def reconstruct_after_erasure(F, x, erased_indices):
    n, m = F.shape
    kept = [i for i in range(m) if i not in erased_indices]
    F_sub = F[:, kept]
    S_sub = F_sub @ F_sub.T
    F_sub_tilde = np.linalg.solve(S_sub, F_sub)
    c_sub = F_sub.T @ x
    x_hat = F_sub_tilde @ c_sub
    return x_hat, np.linalg.norm(x_hat - x)

def tight_erasure_error_formula(F, x, erased_indices, A):
    erased = np.array(erased_indices)
    c_erased = F[:, erased].T @ x
    error = -(1/A) * (F[:, erased] @ c_erased)
    return error

f1 = np.array([1.0, 0.0])
f2 = np.array([-0.5, np.sqrt(3)/2])
f3 = np.array([-0.5, -np.sqrt(3)/2])
F_tri = np.column_stack([f1, f2, f3])

F_sq = regular_polygon(4)
F_pent = regular_polygon(5)

## Lab Exercise 10.1: Erasure Numbers

Compute `erasure\_number(F)` for (a) triangular, (b) square, (c) pentagon, (d) ONB. Compare to $m-n$. Identify full-spark frames.

In [ ]:
F_onb = np.column_stack([np.array([1.0,0.0]), np.array([0.0,1.0])])

frames = [
    ("Triangular", F_tri), ("Square", F_sq), ("Pentagon", F_pent), ("ONB {e1,e2}", F_onb),
]

for name, F in frames:
    n, m = F.shape
    eps = None  # TODO: erasure_number(F)
    full_spark = (eps == m - n)
    print(f"{name}: erasure_number={eps}, m-n={m-n}, full-spark={full_spark}")

## Lab Exercise 10.2: Sub-Frame Reconstruction

Using the triangular frame and $x=(3,-1)^T$: (a) reconstruct after each single erasure. (b) compute the tight-formula error and verify it matches $-\tfrac1A\langle x, f_j \ranglef_j$. (c) which erasure is cheapest?

In [ ]:
x = np.array([3.0, -1.0])
A_tri = 1.5

print("=== Part (a): exact sub-frame reconstruction ===")
for j in range(3):
    x_hat, err = None, None  # TODO: reconstruct_after_erasure(F_tri, x, [j])
    print(f"Erase f{j+1}: error = {err:.2e}")

print("\n=== Part (b): tight-formula error (missing coefficient set to zero) ===")
errors_b = []
for j in range(3):
    err_vec = None  # TODO: tight_erasure_error_formula(F_tri, x, [j], A_tri)
    err_norm = np.linalg.norm(err_vec)
    errors_b.append(err_norm)
    print(f"Erase f{j+1}: error vector = {err_vec.round(4)}, norm = {err_norm:.4f}")

print(f"\n=== Part (c): cheapest erasure ===")
cheapest = np.argmin(errors_b)
print(f"Cheapest erasure: f{cheapest+1} (error={errors_b[cheapest]:.4f})")

*Your explanation: why is this erasure the cheapest, in terms of the coefficient's information content about x?* (replace this text)

## Lab Exercise 10.3: The Square's Weakness

Run `subframe\_spans` on all 6 size-2 erasure patterns of the square. Identify the two failing patterns. Construct a rotated modified frame; verify full-spark, check tightness.

In [ ]:
print("=== All 6 size-2 erasure patterns for the square ===")
for pair in combinations(range(4), 2):
    spans = None  # TODO: subframe_spans(F_sq, list(pair))
    print(f"Erase {pair}: spans = {spans}")

angles_sq = np.array([0, np.pi/2, np.pi, 3*np.pi/2])
angles_modified = angles_sq.copy()
angles_modified[2] += 0.15
angles_modified[3] += 0.08
F_sq_modified = np.vstack([np.cos(angles_modified), np.sin(angles_modified)])

eps_modified = None  # TODO: erasure_number(F_sq_modified)
print(f"\nModified square: erasure_number = {eps_modified} (full-spark target: 2)")

tight_modified, _, _ = None, None, None  # TODO: is_etf(F_sq_modified)
print(f"Modified square still tight? {tight_modified}")

*Your answer: what's special about the two failing erasure patterns for the original square?* (replace this text)

## Lab Exercise 10.4: Worst-Case Erasure Error

For triangle, square, pentagon and a fixed random $x$: (a) worst single-erasure error. (b) worst double-erasure error (spanning patterns only). (c) which achieves the smallest worst-case, and does it have smallest coherence? (Note: comparing different m is exploratory, not what Theorem 10.4.1 addresses directly.)

In [ ]:
rng = np.random.default_rng(11)
x_fixed = rng.standard_normal(2)

frames_A = [("Triangle", F_tri, 1.5), ("Square", F_sq, 2.0), ("Pentagon", F_pent, 2.5)]

print("=== Part (a): worst-case single-erasure error ===")
for name, F, A in frames_A:
    m = F.shape[1]
    errors_single = []
    for j in range(m):
        err_vec = tight_erasure_error_formula(F, x_fixed, [j], A)
        errors_single.append(np.linalg.norm(err_vec))
    worst_single = None  # TODO: max(errors_single)
    print(f"{name}: worst single-erasure error = {worst_single:.4f}")

print("\n=== Part (b): worst-case double-erasure error (spanning patterns only) ===")
worst_doubles = {}
for name, F, A in frames_A:
    m = F.shape[1]
    errors_double = []
    for pair in combinations(range(m), 2):
        if not subframe_spans(F, list(pair)):
            continue
        err_vec = tight_erasure_error_formula(F, x_fixed, list(pair), A)
        errors_double.append(np.linalg.norm(err_vec))

    # NOTE: for a frame with m - n < 2 (e.g. the triangle: m=3, n=2),
    # erasing any 2 vectors leaves only 1, which can never span R^2, so
    # errors_double can be EMPTY. Handle that case explicitly -- do not
    # call max() on an empty list.
    if len(errors_double) == 0:
        worst_double = None
    else:
        worst_double = None  # TODO: max(errors_double)

    worst_doubles[name] = worst_double
    if worst_double is None:
        print(f"{name}: no spanning double-erasure patterns exist "
              f"(m-n={m - F.shape[0]} < 2, cannot survive 2 erasures)")
    else:
        print(f"{name}: worst double-erasure error = {worst_double:.4f}  "
              f"(over {len(errors_double)} spanning patterns)")

print("\n=== Part (c): smallest worst-case, and coherence comparison ===")
comparable = {name: val for name, val in worst_doubles.items() if val is not None}
if comparable:
    best_name = min(comparable, key=comparable.get)
    print(f"Smallest worst-case double-erasure error (among comparable frames): {best_name}")
else:
    print("No frames survive any double erasure -- no comparison possible.")
for name, F, A in frames_A:
    print(f"{name}: coherence = {coherence(F):.4f}")

*Your observation: does the frame with smallest worst-case double-erasure error also have the smallest coherence? Remember these frames have different m -- is this a fair comparison per Theorem 10.4.1?* (replace this text)

## Lab Exercise 10.5 (Challenge): Full-Spark Random Frames

Generate 100 random unit-norm frames for $(n,m)=(2,5)$ and $(3,6)$; compute the fraction achieving full-spark. Explain briefly why full-spark is generic but not universal.

In [ ]:
def random_unit_norm_frame(n, m, seed):
    rng = np.random.default_rng(seed)
    V = rng.standard_normal((n, m))
    return V / np.linalg.norm(V, axis=0)

for n, m in [(2, 5), (3, 6)]:
    full_spark_count = 0
    for seed in range(100):
        F_rand = random_unit_norm_frame(n, m, seed)
        eps = None  # TODO: erasure_number(F_rand)
        if eps == m - n:
            full_spark_count += 1
    print(f"(n,m)=({n},{m}): {full_spark_count}/100 random frames are full-spark "
          f"({full_spark_count}%)")

*Your explanation: why is full-spark generic (almost sure for random frames) but not universal (e.g. the square fails)?* (replace this text)